# Lesson 03 - Mga Pattern ng Agentic Design

Sa araling ito, tatalakayin natin ang tatlong pangunahing pattern ng disenyo para sa paggawa ng epektibong mga AI agent:

1. **Malinaw na Instruksyon para sa Agent** — Paggawa ng tiyak at malinaw na mga prompt na nagtatalaga ng papel para gabayan ang kilos ng agent
2. **Estrukturadong Output gamit ang Pydantic Models** — Pagtiyak na ang mga agent ay magbibigay ng inaasahan at na-validate na data
3. **Mga Agent na May Isang Responsibilidad Lamang** — Pagdidisenyo ng mga nakatuong agent na mahusay sa isang gawain

Ipapatupad natin ang bawat pattern sa isang **scenario ng tagapayo ng destinasyon sa paglalakbay**, unti-unting bumubuo ng sistema na kayang magmungkahi ng mga destinasyon, mag-check ng availability, at mag-asikaso ng mga logistics.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: Malinaw na Mga Tagubilin para sa Ahente

Ang pinakamabisang pattern ay ang pinakamadali rin: pagsulat ng malinaw, detalyadong mga tagubilin para sa iyong ahente.

Magandang mga tagubilin ang naglalarawan:
- **Sino** ang ahente (persona at tono)
- **Ano** ang dapat gawin nito (mga hakbang-hakbang na responsibilidad)
- **Paano** ito dapat kumilos (mga limitasyon at estilo)

Sa ibaba, gagawa tayo ng isang travel concierge agent na may malinaw na mga tagubilin na humuhubog sa bawat tugon na kanyang ginagawa.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Istrakturadong Output gamit ang mga Pydantic Models

Ang free-form na teksto ay kapaki-pakinabang para sa usapan, ngunit ang mga downstream na sistema ay nangangailangan ng istrakturadong data.
Sa pamamagitan ng pagpares ng **mga Pydantic model** sa isang **tool function**, maaari nating:

- Tukuyin ang eksaktong iskema para sa output ng ahente
- Awtomatiko na i-validate ang mga sagot
- Maayos na isama ang mga resulta ng ahente sa lohika ng aplikasyon

Nagpapakilala rin kami ng isang tool na nagbabalik ng mga detalye ng destinasyon upang maging batayan ng ahente ang mga rekomendasyon nito sa totoong data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Mga Ahenteng may Isa lamang na Responsibilidad

Nakikinabang ang mga komplikadong gawain sa paghahati ng trabaho sa iba't ibang pokus na mga ahente, bawat isa ay may iisang responsibilidad:

- Isang **Eksperto sa Destinasyon** na may alam tungkol sa mga lugar at availability
- Isang **Tagaplano ng Lohistika** na humahawak ng mga flight, hotel, at itineraryo

Ito ay sumasalamin sa prinsipyo ng software engineering na *paghiwalay ng mga responsibilidad* — ang bawat ahente ay mas madaling subukan, panatilihin, at pagbutihin nang nakapag-iisa.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Buod

Sa araling ito, inilapat namin ang tatlong disenyo ng agentic sa isang senaryo ng tagapagrekomenda ng paglalakbay:

| Pattern | Pangunahing Ideya | Benepisyo |
|---|---|---|
| **Malinaw na Mga Tagubilin** | Tukuyin ang persona, mga responsibilidad, at mga limitasyon mula sa simula | Pare-pareho, ayon sa tatak na kilos ng ahente |
| **Nakasalalay na Output** | Gamitin ang mga modelong Pydantic bilang format ng tugon | Napatunayan, nababasang makina na mga resulta |
| **Isang Responsibilidad** | Bigyan ang bawat ahente ng isang pokus na gawain | Mas madaling subukan, panatilihin, at pagsamahin |

Ang mga pattern na ito ay kusang bumubuo — maaari mong pagsamahin ang malinaw na mga tagubilin sa nakasalalay na output sa loob ng isang ahente na may isang responsibilidad upang bumuo ng matibay, handa sa produksyon na mga sistema.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
